In [79]:
#importlibraries

#libraries to extract frame
import os
import pandas as pd
import numpy as np
import shutil

#libraries to undergo differencing
from PIL import Image
import numpy as np
import matplotlib.cm as cm
import matplotlib.pyplot as plt
from torchvision.utils import save_image


#set directories
image_to_vid_file = '../../data/Kakadu_fish/location_data/location_dat.csv'
#training data location

training_dat_loc = '../../data/Kakadu_fish/training_data/'

val_frame_dat = pd.read_csv(image_to_vid_file)



In [80]:
def trim_bbox_labels(labels, img_width, img_height):
    """
    Trim the bounding box labels so they fit within an image of size 1080 by 1920.
    
    Args:
    - labels: a list of strings, where each string represents a YOLO bounding box label in the format "class x_center y_center width height"
    - img_width: an integer representing the width of the image in pixels
    - img_height: an integer representing the height of the image in pixels
    
    Returns:
    - a list of strings representing the trimmed YOLO bounding box labels in the same format as the input labels
    """
    new_labels = []
    for label in labels:
        class_id, x_center, y_center, width, height = label.split()
        x_center = float(x_center) * img_width
        y_center = float(y_center) * img_height
        width = float(width) * img_width
        height = float(height) * img_height
        
        x_min = x_center - (width / 2)
        x_max = x_center + (width / 2)
        y_min = y_center - (height / 2)
        y_max = y_center + (height / 2)
        
        # Trim the bounding box coordinates to fit within the image
        if x_min < 0:
            width += x_min
            x_min = 0
        if x_max > img_width:
            width -= (x_max - img_width)
            x_max = img_width
        if y_min < 0:
            height += y_min
            y_min = 0
        if y_max > img_height:
            height -= (y_max - img_height)
            y_max = img_height
        
        # Update the label string with the trimmed coordinates
        new_x_center = (x_min + x_max) / (2 * img_width)
        new_y_center = (y_min + y_max) / (2 * img_height)
        new_width = width / img_width
        new_height = height / img_height
        
        new_label = f"{class_id} {new_x_center} {new_y_center} {new_width} {new_height}"
        new_labels.append(new_label)
    
    return new_labels

In [77]:
#do it for one image
im_num = 15
dat = pd.read_csv(training_dat_loc+'old_labels/'+ val_frame_dat['split'][im_num] + '/' + val_frame_dat['image_name'][im_num].split('.')[0] + '.txt',header=None)  

row_nums = range(0,dat.size)

for row_num in row_nums:
    dat.iloc[row_num] = trim_bbox_labels(dat.iloc[row_num],img_width=1080,img_height=1920)

new_path = training_dat_loc+'fixed_labels/'+ val_frame_dat['split'][im_num] + '/' + val_frame_dat['image_name'][im_num].split('.')[0] + '.txt'


dat.to_csv(new_path, header=None, index=None, sep='\t')

In [81]:
#now lets loop it:
im_num_list = list(range(0,val_frame_dat.shape[0]))

for im_num in im_num_list:
    dat = pd.read_csv(training_dat_loc+'old_labels/'+ val_frame_dat['split'][im_num] + '/' + val_frame_dat['image_name'][im_num].split('.')[0] + '.txt',header=None)  

    row_nums = range(0,dat.size)

    for row_num in row_nums:
        dat.iloc[row_num] = trim_bbox_labels(dat.iloc[row_num],img_width=1080,img_height=1920)

    new_path = training_dat_loc+'fixed_labels/'+ val_frame_dat['split'][im_num] + '/' + val_frame_dat['image_name'][im_num].split('.')[0] + '.txt'

    dat.to_csv(new_path, header=None, index=None, sep='\t')